In [1]:
import json
import logging
import re
import time
import warnings
import os
from pathlib import Path
import sys
from typing import Dict
import pandas as pd
import tiktoken
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_fireworks import ChatFireworks
from tqdm import tqdm

warnings.filterwarnings("ignore")
_ = load_dotenv()
warnings.filterwarnings("ignore", category=ResourceWarning, message="unclosed.*<aiohttp.client.ClientSession.*")

# Redirect logger to use tqdm.write to keep the progress bar clean
class TqdmLoggingHandler(logging.Handler):
    def emit(self, record):
        try:
            msg = self.format(record)
            tqdm.write(msg)
        except Exception:
            self.handleError(record)

logging.basicConfig(level=logging.INFO, handlers=[TqdmLoggingHandler()])
logger = logging.getLogger("debug_logs")

ENCODER = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    return len(ENCODER.encode(text))

In [2]:
prompt = """
You are an expert medical assistant.
You will be provided with a patient's information in a visit and the ordered medications, lab tests and imagings in this visit.

Think about the ordered services and their purposes, and whether they align with the diagnosis. Be strict, do not make speculations on the patient's
information and history beyond the what you're provided with, stick to the explicit information that you have.

Given the patient's case and diagnosis, and using your medical knowledge, decide for each requested service whether it's "Approved" (when it's medically
necessary, for treatment or necessary pain relief, or to rule out possible risk factors, etc..), or "Rejected" (when not medically justified).

Then, only for the services that you reject, you are supposed to recommend a valid alternative service that would be a better fit for this patient's case. IF the service is medically justified but does not align with the provided diagnosis,suggest a more suitable ICD-10 code that would justify the service. You can also recommend for the doctor what to include in his diagnosis
Else if there is no valid alternative, or there is another ordered service that serves the same purpose, just say that.

Return the result in a JSON format that looks like:
Rejected:
{"127658": medicine is not indicated in this case of ..., Do you suspect ...? Please note it in your diagnosis},
{"135987": imaging is not needed, You have ordered service x that serves the same purpose}

Even if two services have the same rejection reason, clarify each of them separately in a key-value pair of its own.
Do not mention or justify any "Approved" services. If all services are approved, return an empty JSON like: Rejected: {}
Clarify your reasons in a friendly advice/recommendation tone.
Return ONLY the raw JSON object. Do not wrap it in markdown code blocks. Your response must start with { and end with }. Do not include ```json or ``` markers.
Return ONLY rejected services, do NOT return any approved services or explain their necessity.
"""

schema = {
    "type": "object",
    "properties": {
        "Rejected": {
            "type": "object",
            "additionalProperties": {
                "type": "object",
                "properties": {
                    "Service ID": {
                        "type": "integer",
                        "description": "Reason for rejecting the service",
                    }
                },
                "required": ["Service ID"],
                "additionalProperties": False,
            },
        }
    },
    "required": ["Rejected"],
    "additionalProperties": False,
}

if 'MODEL_INSTANCE' in globals():
    logger.info("MODEL_INSTANCE already exists.")
else:
    MODEL_INSTANCE = ChatFireworks(
    model="accounts/fireworks/models/deepseek-v3p1",
    temperature=0.0,
    max_tokens=1500, 
    model_kwargs={"top_k": 1, "stream": True},
    request_timeout=(120, 120),
).bind(response_format={"type": "json_object", "schema": schema})

ERROR:asyncio:Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7fe6d96c8160>
ERROR:asyncio:Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7fe6d96ca170>


In [3]:
# …existing code…

def dev_response(info, services, model="accounts/fireworks/models/deepseek-v3p1"):
    """
    Makes predictions on all services in a visit and returns only the
    rejected ones and their rejection reason.

    Args:
        info (dict): patient information (age, gender, diagnosis…)
        services (dict): services in the visit, keyed by VisitServiceID
        model (str): model name (used for logging only)

    Returns:
        elapsed (float): response time
        response (str): model output text
        input_tokens (int)
        output_tokens (int)
    """
    try:
        # reuse the single, already‑bound instance
        global MODEL_INSTANCE
        json_model = MODEL_INSTANCE

        chat_history = [
            SystemMessage(content=prompt),
            HumanMessage(content="Patient Information " + str(info)),
            HumanMessage(content="Ordered Services: " + str(services)),
        ]

        input_text = prompt + str(info) + str(services)
        input_tokens = count_tokens(input_text)

        start = time.time()
        stream = json_model.stream(chat_history)

        full_response = ""
        for chunk in stream:
            if getattr(chunk, "content", None):
                # accumulate – no logging of individual chunks
                full_response += chunk.content
        end = time.time()

        elapsed = end - start
        output_tokens = count_tokens(full_response)

        # log the model name from the argument, not from the runnable
        logger.info(f"[MODEL NAME] {model}")
        logger.info(f"[TOKENS] in={input_tokens} out={output_tokens}")

        return elapsed, full_response, input_tokens, output_tokens

    except Exception as e:
        logger.exception(f"Error generating AI response: {e}")
        raise


def validate_keys(multiple):
    """
    Checks that each key in responses dictionary do not contain multiple services if keys are returned as strings.
    If any was found, it gets split into multiple items each representing one VisitServiceID.

    Args:
    - multiple (list[int]): A dictionary VisitServiceID's as keys and rejection reasons as values.

    Returns:
    - multiple (list[int]): The input dictionary after processing.
    """
    try:
        to_del = []
        temp = {}

        for k, v in multiple.items():
            if not isinstance(k, str):
                continue

            keys = k.split(",")
            if len(keys) > 1:
                logger.debug(f"Found multi-key entry: {k}")
                to_del.append(k)
                for i in keys:
                    temp[i.strip()] = v  # strip whitespace if present

        multiple.update(temp)
        for ele in to_del:
            del multiple[ele]

        return multiple

    except Exception as e:
        logger.exception(f"Error validating keys: {e}")
        raise


def clean_llm_json(response: str) -> dict:
    """Parse JSON from LLM, handling markdown code fences."""
    # Remove markdown code fences if present
    cleaned = re.sub(
        r"^```(?:json)?\s*|\s*```$", "", response.strip(), flags=re.MULTILINE
    )
    return cleaned


def validate_outcome(rejected):
    """
    To prevent recent hallucinations where the model output contains reasoning for approved services which
    get marked as approved by mistake, manually remove any service that has 'approved' in its reasoning.
    """
    to_del = []
    for key, value in rejected.items():
        value_str = str(value)
        if bool(re.search(r'\bapproved\b', value_str, re.IGNORECASE)):
            to_del.append(key)
    for key in to_del:
        del rejected[key]
    return rejected


def duplicate_services(duplications):
    predictions = {}
    for s in duplications:
        predictions[s] = "Duplicated Service"
    return predictions


def request_loop(df):
    responses: Dict[int, str] = {}
    total_time: list[float] = []
    failed_visits: list[int] = []
    total_input_tokens = 0
    total_output_tokens = 0
    visits = df["VisitID"].unique()

    for v in tqdm(visits, desc="Processing"):
        if (
            df.loc[df["VisitID"] == v, "ICD10"].isnull().any()
        ):  # Auto reject visits without a diagnose to avoid unnecessary requests
            responses.update(
                {
                    service: "Diagnosis was not found"
                    for service in df.loc[df["VisitID"] == v, "VisitServiceID"]
                }
            )
        else:
            try:
                row = df[df["VisitID"] == v].iloc[0]
                # Select only features that are not null to avoid entering empty problem note and symptoms to the model
                selected_cols = [
                    col
                    for col in row.index
                    if pd.notna(row[col])
                    and col
                    in [
                        "AGE",
                        "PATIENT_GENDER",
                        "CHIEF_COMPLAINT",
                        "PROVIDER_DEPARTMENT",
                        "ICD10",
                        "DIAGNOS_NAME",
                        "ProblemNote",
                        "Symptoms",
                    ]
                ]
                p_info = row[selected_cols].to_dict()
                v_services = df.loc[
                    df["VisitID"] == v,
                    ["VisitServiceID", "Service_Name", "Quantity"],
                ].set_index("VisitServiceID")

                if df.loc[df["VisitID"] == v, "Visit_Type"].iloc[0] == "Outpatient":
                    # Auto reject duplicated services to avoid unnecessary input in requests in case of outpatient only
                    v_services = v_services.drop_duplicates(keep="first")
                    dups = list(
                        set(df.loc[df["VisitID"] == v, "VisitServiceID"])
                        - set(v_services.index)
                    )
                    if dups:
                        responses.update(duplicate_services(dups))

                res_time, answer ,in_tokens, out_tokens = dev_response(
                    p_info, v_services.to_dict(orient="index")
                )
                total_input_tokens += in_tokens
                total_output_tokens += out_tokens

                logger.debug(
                    f"Response received in {res_time:.2f} seconds for visit {v}"
                )
                logger.info(f"[TOKENS] Visit={v} | in={in_tokens} | out={out_tokens} | total={in_tokens + out_tokens}")
                total_time.append(res_time)

            except (
                Exception
            ) as e:  # Errors due to API provider, server busy for example
                logger.error(f"Error processing visit {v}: {e}")
                failed_visits.append(v)
                time.sleep(60)
                continue

            try:
                rejections = validate_outcome(json.loads(answer).get("Rejected", {}))
                responses.update(rejections)
            except (
                json.JSONDecodeError
            ):  # Errors due to JSON parsing, JSON schema error or unterminated string for example
                try:
                    cleaned = clean_llm_json(answer)
                    rejections = validate_outcome(json.loads(cleaned).get("Rejected", {}))
                    responses.update(rejections)
                except json.JSONDecodeError as e:
                    failed_visits.append(v)
                    logger.error(
                        f"Failed to parse JSON response for visit {v}: {e}, Raw response: {answer}"
                    )
    return responses, total_time, failed_visits, total_input_tokens, total_output_tokens


def make_preds(df):
    logger.info(
        f"Query returned {len(df)} rows with {df['VisitID'].nunique()} unique visits"
    )
    responses, total_time, failed_visits, total_input_tokens, total_output_tokens = request_loop(df)

    try:
        if failed_visits:
            logger.debug(f"Retrying on Failed Visits: {failed_visits}")

            failed_responses, failed_total_time, failed_visits, retry_input, retry_output = request_loop(
                df[df["VisitID"].isin(failed_visits)]
            )

            if failed_responses:
                responses.update(failed_responses)

            total_time = total_time + failed_total_time

            total_input_tokens += retry_input
            total_output_tokens += retry_output

    except Exception as e:
        logger.debug(f"Error running failed visits: {e}")

    logger.debug(f"Failed Visits: {failed_visits}")
    logger.info(f"Inference time: {sum(total_time):.2f} seconds")

    total_tokens = total_input_tokens + total_output_tokens

    INPUT_PRICE_PER_TOKEN = 0.56 / 1_000_000
    OUTPUT_PRICE_PER_TOKEN = 1.68 / 1_000_000

    total_cost = (
        total_input_tokens * INPUT_PRICE_PER_TOKEN +
        total_output_tokens * OUTPUT_PRICE_PER_TOKEN
    )


    #df.drop(df[df["VisitID"].isin(failed_visits)].index, inplace=True)

    metrics = {
        "total_input_tokens": total_input_tokens,
        "total_output_tokens": total_output_tokens,
        "total_tokens": total_tokens,
        "total_cost": total_cost,
    }

    history_df = write_preds(validate_keys(responses), df, failed_visits)

    return history_df, metrics


def write_preds(responses, df, failed_visits):
    df["VisitServiceID"] = df["VisitServiceID"].astype(int)
    reason_col = "Reason/Recommendation"

    # Helper to find an existing column from common alternatives
    def choose_col(candidates):
        for c in candidates:
            if c in df.columns:
                return c
        return None

    # Common alternative names in incoming data
    diagnose_col = choose_col(["Diagnose", "DIAGNOS_NAME", "Diagnose_Name", "DIAGNOSIS", "Diagnosis"]) 
    chief_col = choose_col(["Chief_Complaint", "CHIEF_COMPLAINT", "ChiefComplaintNotes", "ChiefComplaint"])

    if responses:
        pred_df = pd.DataFrame(
            [
                {
                    "VisitServiceID": int(k),
                    reason_col: v,
                    "Medical_Prediction": "Rejected",
                }
                for k, v in responses.items()
            ]
        )
        logger.debug(f"Created prediction dataframe with {len(pred_df)} rejected services")
        df = df.merge(pred_df, on="VisitServiceID", how="left")
    else:
        logger.info("No rejected services found, all will be marked as Approved")
        df["Medical_Prediction"] = "Approved"
        df[reason_col] = None

    df["Medical_Prediction"] = df["Medical_Prediction"].fillna("Approved")
    # If a VisitID is in the failed list, label it as "Failed to reach LLM"
    df.loc[df["VisitID"].isin(failed_visits), "Medical_Prediction"] = "Failed to reach LLM"
    df.loc[df["VisitID"].isin(failed_visits), reason_col] = "API Error or Timeout"

    # Ensure standardized output columns exist. If alternatives exist, copy them into the standard names.
    if diagnose_col and diagnose_col != "Diagnose":
        df["Diagnose"] = df[diagnose_col]
    elif "Diagnose" not in df.columns:
        df["Diagnose"] = None

    if chief_col and chief_col != "Chief_Complaint":
        df["Chief_Complaint"] = df[chief_col]
    elif "Chief_Complaint" not in df.columns:
        df["Chief_Complaint"] = None

    # Ensure other expected columns exist so selection doesn't fail
    for col in ["Service_Name", "ProblemNote", "Symptoms"]:
        if col not in df.columns:
            df[col] = None

    history_df = df[
        [
            "VisitID",
            "VisitServiceID",
            "Service_Name",
            "Medical_Prediction",
            reason_col,
            "Diagnose",
            "Chief_Complaint",
            "ProblemNote",
            "Symptoms",
        ]
    ]

    logger.info(f"Final prediction dataframe has {len(history_df)} rows")

    return history_df

In [4]:
def process_visits_batched(df, batch_size=10, checkpoint_dir="/home/ai/Workspace/AmrJr/eligibility-etl-airflow/data/checkpoints"):
    """
    Process visits in batches with checkpointing for resume capability.
    
    Args:
        df: DataFrame with visits
        batch_size: Number of unique visits per batch
        checkpoint_dir: Directory to save batch checkpoints
        
    Returns:
        final_report: Combined results from all batches
        metrics: Aggregated metrics
    """
    Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)
    
    visits = df["VisitID"].unique()
    total_visits = len(visits)
    
    # Create batches
    visit_batches = [visits[i:i + batch_size] for i in range(0, total_visits, batch_size)]
    num_batches = len(visit_batches)
    
    logger.info(f"Total visits: {total_visits}, Batch size: {batch_size}, Total batches: {num_batches}")
    
    # Track overall results
    all_responses = {}
    all_times = []
    all_failed_visits = []
    total_input_tokens = 0
    total_output_tokens = 0
    
    # Process each batch
    for batch_idx, batch_visits in enumerate(tqdm(visit_batches, desc="Batches")):
        checkpoint_file = os.path.join(checkpoint_dir, f"batch_{batch_idx}_checkpoint.json")
        
        # Skip if batch already completed
        if os.path.exists(checkpoint_file):
            logger.info(f"Batch {batch_idx} already processed, loading from checkpoint")
            with open(checkpoint_file, 'r') as f:
                batch_data = json.load(f)
                all_responses.update(batch_data.get('responses', {}))
                all_times.extend(batch_data.get('times', []))
                all_failed_visits.extend(batch_data.get('failed_visits', []))
                total_input_tokens += batch_data.get('input_tokens', 0)
                total_output_tokens += batch_data.get('output_tokens', 0)
            continue
        
        # Process this batch
        batch_df = df[df["VisitID"].isin(batch_visits)]
        logger.info(f"Processing batch {batch_idx + 1}/{num_batches} with {len(batch_visits)} visits")
        
        responses, times, failed_visits, in_tokens, out_tokens = request_loop(batch_df)
        
        # Save batch checkpoint
        checkpoint_data = {
            'batch_idx': batch_idx,
            'visits': batch_visits.tolist(),
            'responses': responses,
            'times': times,
            'failed_visits': failed_visits,
            'input_tokens': in_tokens,
            'output_tokens': out_tokens,
        }
        
        with open(checkpoint_file, 'w') as f:
            json.dump(checkpoint_data, f, indent=2)
        
        logger.info(f"Batch {batch_idx + 1} checkpoint saved")
        
        # Aggregate results
        all_responses.update(responses)
        all_times.extend(times)
        all_failed_visits.extend(failed_visits)
        total_input_tokens += in_tokens
        total_output_tokens += out_tokens
    
    # Retry failed visits in batches if any
    if all_failed_visits:
        logger.info(f"Retrying {len(all_failed_visits)} failed visits in batches")
        unique_failed = list(set(all_failed_visits))
        failed_batches = [unique_failed[i:i + batch_size] for i in range(0, len(unique_failed), batch_size)]
        
        for batch_idx, batch_visits in enumerate(tqdm(failed_batches, desc="Retry Batches")):
            retry_df = df[df["VisitID"].isin(batch_visits)]
            responses, times, _, in_tokens, out_tokens = request_loop(retry_df)
            
            if responses:
                all_responses.update(responses)
            all_times.extend(times)
            total_input_tokens += in_tokens
            total_output_tokens += out_tokens
    
    # Compute final metrics
    total_tokens = total_input_tokens + total_output_tokens
    INPUT_PRICE_PER_TOKEN = 0.56 / 1_000_000
    OUTPUT_PRICE_PER_TOKEN = 1.68 / 1_000_000
    total_cost = (total_input_tokens * INPUT_PRICE_PER_TOKEN +
                  total_output_tokens * OUTPUT_PRICE_PER_TOKEN)
    
    metrics = {
        "total_input_tokens": total_input_tokens,
        "total_output_tokens": total_output_tokens,
        "total_tokens": total_tokens,
        "total_cost": total_cost,
        "num_batches": num_batches,
        "batch_size": batch_size,
    }
    
    # Generate final report
    history_df = write_preds(validate_keys(all_responses), df, all_failed_visits)
    
    logger.info(f"Batch processing complete. Total time: {sum(all_times):.2f} seconds")
    
    return history_df, metrics


In [5]:
# 1. Load your CSV
raw_df = pd.read_csv("/home/ai/Workspace/AmrJr/eligibility-etl-airflow/data/x_test (2).csv")

# 2. Combine ICD codes (from your logic)
diag_cols = ['ICDDiagnoseCode1', 'ICDDiagnoseCode2', 'ICDDiagnoseCode3']
existing_diag_cols = [c for c in diag_cols if c in raw_df.columns]
raw_df['Combined_ICD'] = raw_df[existing_diag_cols].apply(lambda x: ', '.join(x.dropna().astype(str)), axis=1)

# 3. Rename columns (corrected - all required columns)
df = raw_df.rename(columns={
    "Specialty": "PROVIDER_DEPARTMENT",
    "PatientEnGender": "PATIENT_GENDER",
    "Age": "AGE",
    "Combined_ICD": "ICD10",
    "ResponseQuantity": "Quantity",
    "Problem_Note": "ProblemNote",
    "ChiefComplaintNotes": "CHIEF_COMPLAINT",
})

# --- MANDATORY COLUMNS FOR YOUR LOGIC ---
# These are the columns your request_loop is looking for:
df['VisitServiceID'] = range(len(df)) 
df['Visit_Type'] = 'Outpatient'        
# ---------------------------------------

# 4. Start Audit with Batch Processing
# Set batch_size to control how many visits are processed at once
# Checkpoints are saved automatically for each batch for resume capability
final_report, metrics = process_visits_batched(df, batch_size=10)


INFO:debug_logs:Total visits: 1747, Batch size: 10, Total batches: 175


Batches:   0%|          | 0/175 [00:00<?, ?it/s]

INFO:debug_logs:Batch 0 already processed, loading from checkpoint
INFO:debug_logs:Batch 1 already processed, loading from checkpoint
INFO:debug_logs:Batch 2 already processed, loading from checkpoint


Batches:   3%|▎         | 6/175 [00:00<00:13, 12.62it/s]

INFO:debug_logs:Batch 3 already processed, loading from checkpoint
INFO:debug_logs:Batch 4 already processed, loading from checkpoint
INFO:debug_logs:Batch 5 already processed, loading from checkpoint
INFO:debug_logs:Batch 6 already processed, loading from checkpoint
INFO:debug_logs:Batch 7 already processed, loading from checkpoint
INFO:debug_logs:Batch 8 already processed, loading from checkpoint


...

Batches:  69%|██████▊   | 120/175 [00:03<00:01, 36.41it/s]

INFO:debug_logs:Batch 117 already processed, loading from checkpoint
INFO:debug_logs:Batch 118 already processed, loading from checkpoint
INFO:debug_logs:Batch 119 already processed, loading from checkpoint
INFO:debug_logs:Batch 120 already processed, loading from checkpoint
INFO:debug_logs:Batch 121 already processed, loading from checkpoint
INFO:debug_logs:Batch 122 already processed, loading from checkpoint
INFO:debug_logs:Batch 123 already processed, loading from checkpoint
INFO:debug_logs:Batch 124 already processed, loading from checkpoint
INFO:debug_logs:Batch 125 already processed, loading from checkpoint
INFO:debug_logs:Batch 126 already processed, loading from checkpoint
INFO:debug_logs:Batch 127 already processed, loading from checkpoint


INFO:debug_logs:Batch 128 already processed, loading from checkpoint
INFO:debug_logs:Batch 129 already processed, loading from checkpoint
INFO:debug_logs:Batch 130 already processed, loading from checkpoint
INFO:debug_logs:Batch 131 already processed, loading from checkpoint
INFO:debug_logs:Batch 132 already processed, loading from checkpoint
INFO:debug_logs:Batch 133 already processed, loading from checkpoint
INFO:debug_logs:Batch 134 already processed, loading from checkpoint
INFO:debug_logs:Batch 135 already processed, loading from checkpoint
INFO:debug_logs:Batch 136 already processed, loading from checkpoint
INFO:debug_logs:Batch 137 already processed, loading from checkpoint
INFO:debug_logs:Batch 138 already processed, loading from checkpoint
INFO:debug_logs:Batch 139 already processed, loading from checkpoint
INFO:debug_logs:Batch 140 already processed, loading from checkpoint
INFO:debug_logs:Batch 141 already processed, loading from checkpoint
INFO:debug_logs:Batch 142 already 

Batches: 100%|██████████| 175/175 [00:04<00:00, 43.31it/s]


INFO:debug_logs:Batch 167 already processed, loading from checkpoint
INFO:debug_logs:Batch 168 already processed, loading from checkpoint
INFO:debug_logs:Batch 169 already processed, loading from checkpoint
INFO:debug_logs:Batch 170 already processed, loading from checkpoint
INFO:debug_logs:Batch 171 already processed, loading from checkpoint
INFO:debug_logs:Batch 172 already processed, loading from checkpoint
INFO:debug_logs:Batch 173 already processed, loading from checkpoint
INFO:debug_logs:Batch 174 already processed, loading from checkpoint
INFO:debug_logs:Final prediction dataframe has 7503 rows
INFO:debug_logs:Batch processing complete. Total time: 3066.84 seconds


In [ ]:
final_report

,VisitID,VisitServiceID,Service_Name,Medical_Prediction,Reason/Recommendation,Diagnose,Chief_Complaint,ProblemNote,Symptoms
0,772175,0,COMPLETE BLOOD COUNT.,Approved,NaN,None,abdominal pain,NaN,NaN
1,762446,1,COMPLETE BLOOD COUNT.,Approved,NaN,None,hematuria,NaN,NaN
2,762446,2,CREATININE IN SERUM,Approved,NaN,None,hematuria,NaN,NaN
3,762446,3,URINE CULTURE & SENSITIVITY,Approved,NaN,None,hematuria,NaN,NaN
4,772175,4,FERRITIN IN SERUM,Rejected,Ferritin in serum is not indicated for abdomin...,None,abdominal pain,NaN,NaN
...,...,...,...,...,...,...,...,...,...
7498,868452,7498,CREATININE IN SERUM,Approved,NaN,None,DYSLIPIDEMIA (AMBULATORY),NaN,NaN
7499,880586,7499,COMPLETE BLOOD COUNT.,Approved,NaN,None,chronic fatigue dizziness bone pain,NaN,"chronic fatigue , dizziness , bone pain"
7500,880586,7500,TRANSFERRIN PROTEIN IN SERUM,Rejected,Transferrin protein in serum is not indicated ...,None,chronic fatigue dizziness bone pain,NaN,"chronic fatigue , dizziness , bone pain"
7501,887675,7501,PAP SMEAR (CERVICAL CYTOLOGY) ( LIQUID BASED C...,Approved,NaN,None,MF2 YR CAOMPLINE OF POSTCOITAL BLEEDING AND I...,MF2 YR CAOMPLINE OF POSTCOITAL BLEEDING AND I...,MF2 YR CAOMPLINE OF POSTCOITAL BLEEDING AND I...


In [7]:
# Validation: compare LLM predictions with ground-truth y_test by Service_Name
import math
from collections import Counter

def _choose_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def validate_predictions(pred_df, y_path, merge_on='Service_Name'):
    # Load ground truth
    y = pd.read_csv(y_path)

    # Determine columns to merge on (prefer exact service name, fallback to VisitServiceID)
    if merge_on not in pred_df.columns or merge_on not in y.columns:
        if 'VisitServiceID' in pred_df.columns and 'VisitServiceID' in y.columns:
            left_key = 'VisitServiceID'
            right_key = 'VisitServiceID'
        else:
            # try fuzzy service name key by normalizing names into a helper column
            left_key = '_merge_key'
            right_key = '_merge_key'
            # Build a Series for pred_df service names (avoid defaulting to a plain string)
            if 'Service_Name' in pred_df.columns:
                left_series = pred_df['Service_Name'].astype(str).str.strip().str.lower()
            else:
                left_series = pd.Series([''] * len(pred_df), index=pred_df.index)

            if 'Service_Name' in y.columns:
                right_series = y['Service_Name'].astype(str).str.strip().str.lower()
            else:
                right_series = pd.Series([''] * len(y), index=y.index)

            pred_df[left_key] = left_series
            y[right_key] = right_series
    else:
        left_key = merge_on
        right_key = merge_on

    # Find true label column in y (common names) and predicted label in pred_df
    true_label_col = _choose_col(y, ['Medical_Prediction', 'True_Label', 'label', 'Target', 'Actual'])
    pred_label_col = _choose_col(pred_df, ['Medical_Prediction', 'Prediction', 'Predicted'])

    if true_label_col is None:
        raise ValueError("Could not find a label column in y (checked common names)")
    if pred_label_col is None:
        raise ValueError("Could not find predicted label column in predictions dataframe")

    merged = pred_df.merge(y, left_on=left_key, right_on=right_key, how='inner', suffixes=("_pred","_true"))

    # Map labels to binary: Rejected -> 1, others -> 0. Treat failures as separate.
    def map_label(x):
        if isinstance(x, str) and 'reject' in x.lower():
            return 1
        return 0

    merged['_y_true_bin'] = merged[true_label_col].apply(map_label)
    merged['_y_pred_bin'] = merged[pred_label_col].apply(map_label)

    # Exclude failed LLM calls from metric calc but report them
    failed_mask = merged[pred_label_col].astype(str).str.contains('Failed to reach LLM', case=False, na=False)
    eval_df = merged[~failed_mask]

    tp = int(((eval_df['_y_true_bin'] == 1) & (eval_df['_y_pred_bin'] == 1)).sum())
    tn = int(((eval_df['_y_true_bin'] == 0) & (eval_df['_y_pred_bin'] == 0)).sum())
    fp = int(((eval_df['_y_true_bin'] == 0) & (eval_df['_y_pred_bin'] == 1)).sum())
    fn = int(((eval_df['_y_true_bin'] == 1) & (eval_df['_y_pred_bin'] == 0)).sum())

    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else math.nan
    precision = tp / (tp + fp) if (tp + fp) > 0 else math.nan
    recall = tp / (tp + fn) if (tp + fn) > 0 else math.nan
    f1 = 2 * precision * recall / (precision + recall) if (not math.isnan(precision) and not math.isnan(recall) and (precision + recall) > 0) else math.nan

    metrics = {
        'n_evaluated': int(len(eval_df)),
        'tp': tp,
        'tn': tn,
        'fp': fp,
        'fn': fn,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'n_failed_llm': int(failed_mask.sum()),
        'n_merged': int(len(merged)),
    }

    return merged, metrics


# Run validation using the y_test file (adjust path if needed)
y_path = '/home/ai/Workspace/AmrJr/eligibility-etl-airflow/data/y_test (2).csv'
try:
    merged_val, val_metrics = validate_predictions(final_report, y_path, merge_on='Service_Name')
    print('Validation metrics:')
    print(val_metrics)
    print('\nSample mismatches:')
    mismatches = merged_val[merged_val['_y_true_bin'] != merged_val['_y_pred_bin']].head(20)
    display_cols = [c for c in ['VisitID', 'VisitServiceID', 'Service_Name_pred', 'Service_Name_true', 'Medical_Prediction_pred', 'Medical_Prediction_true'] if c in merged_val.columns]
    print(mismatches[display_cols].head(20))
except Exception as e:
    print('Validation failed:', e)


Validation failed: Could not find a label column in y (checked common names)


In [9]:
output_path = "/home/ai/Workspace/AmrJr/eligibility-etl-airflow/data/audit_results.csv"
final_report.to_csv(output_path, index=False)
tqdm.write(f"\nResults saved to: {output_path}")
tqdm.write(f"\nFirst 20 rows:")



Results saved to: /home/ai/Workspace/AmrJr/eligibility-etl-airflow/data/audit_results.csv

First 20 rows:


In [10]:
# ── Append y_test ground truth to final_report ────────────────────────────────
y_test = pd.read_csv("/home/ai/Workspace/AmrJr/eligibility-etl-airflow/data/y_test (2).csv")

# Normalize both keys for a safe join
for col in ['Service_Name']:
    final_report[col + '_key'] = final_report[col].astype(str).str.strip().str.lower()
    y_test[col + '_key']       = y_test[col].astype(str).str.strip().str.lower()

# Ensure VisitID is the same type on both sides
final_report['VisitID'] = final_report['VisitID'].astype(str).str.strip()
y_test['VisitID']       = y_test['VisitID'].astype(str).str.strip()

# Merge on both VisitID + normalized Service_Name
final_report_with_truth = final_report.merge(
    y_test[['VisitID', 'Service_Name_key', 'Medical_Prediction']],
    on=['VisitID', 'Service_Name_key'],
    how='left',
    suffixes=('_pred', '_true')   # → Medical_Prediction_pred / Medical_Prediction_true
)

print(final_report_with_truth.shape)
print(final_report_with_truth.head())

KeyError: 'Service_Name'